In [ ]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time 6CpRuruT0bcAVw6vsqQvA8tm

In [ ]:
import crunch
# crunch_tools = crunch.load_notebook()   # uncomment for local test / submit

In [ ]:
import os, math
from bisect import bisect_left, bisect_right
from collections import deque
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier


In [ ]:
# ---- helpers computed once per series on the reference ------------------


def _autocorr(x, lag):
    if len(x) <= lag:
        return 0.0
    a, b = x[:-lag], x[lag:]
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0


def _fit_ar(x, p=5):
    """Least-squares AR(p) with intercept. Returns (coefs, resid_var)."""
    if len(x) <= p + 5:
        return None, 1.0
    rows = np.lib.stride_tricks.sliding_window_view(x, p + 1)
    Y = rows[:, -1]
    Xd = np.column_stack([np.ones(len(rows)), rows[:, :-1][:, ::-1]])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None)
    resid = Y - Xd @ beta
    return beta, float(resid.var()) if len(resid) else 1.0


def _norm_psd(seg, win):
    """Hann-windowed one-sided PSD, normalised to sum 1. None if degenerate."""
    x = (seg - seg.mean()) * win
    sp = np.fft.rfft(x)
    p = (sp.real ** 2 + sp.imag ** 2)[1:]   # drop DC
    s = p.sum()
    if s <= 0 or not np.isfinite(s):
        return None
    return p / s


def _spec_ec(p, freqs, logk):
    ent = -float((p * np.log(p + 1e-12)).sum()) * logk
    cen = float((freqs * p).sum())
    return ent, cen

In [ ]:
# Streaming O(1)/O(window) feature extractor + CatBoost train/infer.
# StreamStateV3 = deployed 43-feature StreamState + 5 shape/changepoint features
# (var_cusum_max, kuiper, cvm, skew_shift, kurt_shift) + variance-GLR (var_glr,
# var_glr_peak): a Gaussian likelihood-ratio scan for a variance change at an
# unknown split. Offline (stream_features_v3.ipynb): CatBoost 0.6007(LGB)->0.6130,
# var_cusum_max gain-rank 9/50, var_glr 15/50. Single CatBoost captures the blend.
# Helpers (_autocorr/_fit_ar/_norm_psd/_spec_ec) are defined in the previous cell.


class StreamStateV3:
    VAR_MIN = 20; SPEC_W = 128; AC_LAGS_NEW = (2, 3, 5, 10); AR_ORDERS_NEW = (2, 10); MAXLAG = 10
    GLR_MIN = 30                                                          # GLR: need enough points
    _BASE = ("n_online", "mean_shift_z", "last_z", "online_logvar", "online_std",
             "online_skew", "online_kurt", "cumsum_range", "cusum_max",
             "tail2_frac", "tail3_frac", "jump_freq", "signrun_freq",
             "ks_stat", "rank_dev", "ac1_online", "ac1_shift", "ar_resid_logratio",
             "ref_log_n", "ref_skew", "ref_kurt", "ref_ac1")
    _SPEC = ("spec_entropy", "spec_entropy_shift", "spec_centroid_shift")
    _EXTRA = ("var_cusum_max", "kuiper", "cvm", "skew_shift", "kurt_shift")
    _GLR = ("var_glr", "var_glr_peak")                                    # GLR: new v3 features

    def __init__(self, bins=16, roll_windows=(50, 200), ewma_alphas=(0.05, 0.2), ar_p=5):
        self.bins = bins; self.rw = tuple(roll_windows); self.alphas = tuple(ewma_alphas); self.p = ar_p
        fn = list(self._BASE)
        for a in self.alphas: fn += ["ewma%.2f_z" % a, "ewma%.2f_logvar" % a]
        for w in self.rw: fn += ["roll%d_meanshift" % w, "roll%d_logvar" % w]
        self._ew_off = len(self._BASE); self._roll_off = self._ew_off + 2 * len(self.alphas)
        self._spec_off = len(fn); fn += list(self._SPEC)
        self._dep_off = len(fn); self._dep_names = []
        for L in self.AC_LAGS_NEW: self._dep_names += ["ac%d_online" % L, "ac%d_shift" % L]
        for pp in self.AR_ORDERS_NEW: self._dep_names += ["ar%d_resid_logratio" % pp]
        fn += self._dep_names
        self._extra_off = len(fn); fn += list(self._EXTRA)
        self._glr_off = len(fn); fn += list(self._GLR)                    # GLR
        self.feature_names = fn; self.ncol = len(fn)
        self._out = np.zeros(self.ncol, dtype=np.float64)
        self._bin_targets = np.linspace(1.0 / self.bins, 1.0, self.bins)
        W = self.SPEC_W; self._spec_win = np.hanning(W)
        self._spec_freqs = np.arange(1, W // 2 + 1, dtype=np.float64) / (W // 2)
        self._spec_logk = 1.0 / math.log(len(self._spec_freqs))
        self._glr_fracs = np.linspace(0.1, 0.9, 12)                       # GLR: candidate split fractions

    def reset(self, reference):
        r = np.asarray(reference, dtype=np.float64)
        self.mu_h = float(r.mean()) if len(r) else 0.0
        self.sd_h = max(float(r.std(ddof=1)) if len(r) > 1 else 1.0, 1e-8)
        self.var_h = self.sd_h ** 2; self.ref_n = len(r); self._inv_refn = 1.0 / max(self.ref_n, 1)
        z = (r - self.mu_h) / self.sd_h if len(r) else r
        self.ref_skew = float((z ** 3).mean()) if len(r) else 0.0
        self.ref_kurt = float((z ** 4).mean() - 3.0) if len(r) else 0.0
        self.ref_ac1 = _autocorr(r, 1); self.ref_log_n = math.log(self.ref_n + 1.0)
        self.ref_ac = {L: _autocorr(r, L) for L in self.AC_LAGS_NEW}
        self.sorted_ref = np.sort(r).tolist() if len(r) else []
        edges = (np.quantile(r, np.linspace(0, 1, self.bins + 1)[1:-1]) if len(r) else np.zeros(self.bins - 1))
        self.edges = edges.tolist()
        self.ar, self.ar_var_h = _fit_ar(r, self.p)
        self.arN = {pp: _fit_ar(r, pp) for pp in self.AR_ORDERS_NEW}
        self.ref_spec_ent = 0.0; self.ref_spec_cen = 0.0
        if len(r) >= self.SPEC_W:
            W = self.SPEC_W; acc = np.zeros(W // 2); cnt = 0
            for s in range(0, len(r) - W + 1, W // 2):
                p = _norm_psd(r[s:s + W], self._spec_win)
                if p is not None: acc += p; cnt += 1
            if cnt:
                pr = acc / acc.sum(); self.ref_spec_ent, self.ref_spec_cen = _spec_ec(pr, self._spec_freqs, self._spec_logk)
        self.n = 0; self.mean = self.M2 = self.M3 = self.M4 = 0.0
        self.sx = self.sxx = self.sxy = 0.0
        self.cmax = self.cmin = self.cumsum = 0.0
        self.cusum_p = self.cusum_n = self.cusum_max = 0.0
        self.cusum_var_p = self.cusum_var_n = self.cusum_var_max = 0.0
        self.pref_s1 = [0.0]; self.pref_s2 = [0.0]; self.var_glr_peak_v = 0.0          # GLR: prefix sums of z, z^2
        self.tail2 = self.tail3 = 0; self.binc = np.zeros(self.bins); self.rank_sum = 0.0
        self.jumps = 0; self.signruns = 0; self.prev = None
        self.ar_resid_ss = 0.0; self.ar_n = 0; self.arbuf = deque(maxlen=self.p)
        self.roll = {w: (deque(maxlen=w), [0.0, 0.0]) for w in self.rw}
        self.ew = {a: self.mu_h for a in self.alphas}; self.ewv = {a: 0.0 for a in self.alphas}
        self.specbuf = deque(maxlen=self.SPEC_W); self._last_spec_ent = 0.0; self._last_spec_cen = 0.0
        self.lagbuf = deque(maxlen=self.MAXLAG)
        self.acP = {L: 0.0 for L in self.AC_LAGS_NEW}; self.acC = {L: 0 for L in self.AC_LAGS_NEW}
        self.arNbuf = {pp: deque(maxlen=pp) for pp in self.AR_ORDERS_NEW}
        self.arNss = {pp: 0.0 for pp in self.AR_ORDERS_NEW}; self.arNn = {pp: 0 for pp in self.AR_ORDERS_NEW}
        return self

    def _var_glr(self, L):                                                # GLR: max Gaussian variance-change LR over candidate splits
        s1, s2 = self.pref_s1, self.pref_s2
        tot1 = s1[L]; tot2 = s2[L]
        v0 = tot2 / L - (tot1 / L) ** 2
        if v0 <= 1e-9: return 0.0
        best = 0.0; lv0 = math.log(v0 + 1e-8)
        for frac in self._glr_fracs:
            k = int(frac * L)
            if k < 8 or k > L - 8: continue
            m1 = s1[k] / k; v1 = s2[k] / k - m1 * m1
            nk = L - k; m2 = (tot1 - s1[k]) / nk; v2 = (tot2 - s2[k]) / nk - m2 * m2
            if v1 <= 1e-9 or v2 <= 1e-9: continue
            g = L * lv0 - k * math.log(v1 + 1e-8) - nk * math.log(v2 + 1e-8)
            if g > best: best = g
        return best

    def update(self, x):
        x = float(x); self.n += 1; n = self.n
        sd_h = self.sd_h; mu_h = self.mu_h; var_h = self.var_h
        d = x - self.mean; dn = d / n; dn2 = dn * dn; term = d * dn * (n - 1)
        self.mean += dn
        self.M4 += term * dn2 * (n * n - 3 * n + 3) + 6 * dn2 * self.M2 - 4 * dn * self.M3
        self.M3 += term * dn * (n - 2) - 3 * dn * self.M2; self.M2 += term
        var = self.M2 / (n - 1) if n > 1 else 0.0
        std = math.sqrt(var) if var > 0 else 0.0
        zx = (x - mu_h) / sd_h
        self.pref_s1.append(self.pref_s1[-1] + zx); self.pref_s2.append(self.pref_s2[-1] + zx * zx)   # GLR
        self.cumsum += zx
        if self.cumsum > self.cmax: self.cmax = self.cumsum
        if self.cumsum < self.cmin: self.cmin = self.cumsum
        cp = self.cusum_p + zx - 0.5; cn = self.cusum_n - zx - 0.5
        self.cusum_p = cp if cp > 0.0 else 0.0; self.cusum_n = cn if cn > 0.0 else 0.0
        if self.cusum_p > self.cusum_max: self.cusum_max = self.cusum_p
        if self.cusum_n > self.cusum_max: self.cusum_max = self.cusum_n
        vinc = zx * zx - 1.0
        vp = self.cusum_var_p + vinc - 0.5; vn = self.cusum_var_n - vinc - 0.5
        self.cusum_var_p = vp if vp > 0.0 else 0.0; self.cusum_var_n = vn if vn > 0.0 else 0.0
        if self.cusum_var_p > self.cusum_var_max: self.cusum_var_max = self.cusum_var_p
        if self.cusum_var_n > self.cusum_var_max: self.cusum_var_max = self.cusum_var_n
        azx = zx if zx >= 0 else -zx
        if azx > 2.0: self.tail2 += 1
        if azx > 3.0: self.tail3 += 1
        if self.prev is not None:
            if abs(x - self.prev) > 2.0 * sd_h: self.jumps += 1
            if (x - mu_h) * (self.prev - mu_h) < 0: self.signruns += 1
            self.sxy += x * self.prev
        self.sx += x; self.sxx += x * x
        lo = bisect_left(self.sorted_ref, x); hi = bisect_right(self.sorted_ref, x)
        self.rank_sum += 0.5 * (lo + hi) * self._inv_refn
        self.binc[bisect_right(self.edges, x)] += 1
        if self.ar is not None and len(self.arbuf) == self.p:
            lag = np.array(self.arbuf, dtype=np.float64)[::-1]; pred = self.ar[0] + self.ar[1:] @ lag
            r_ = x - pred; self.ar_resid_ss += r_ * r_; self.ar_n += 1
        self.arbuf.append(x)
        lb = self.lagbuf
        for L in self.AC_LAGS_NEW:
            if len(lb) >= L: self.acP[L] += x * lb[-L]; self.acC[L] += 1
        lb.append(x)
        for pp in self.AR_ORDERS_NEW:
            beta, _vh = self.arN[pp]; buf = self.arNbuf[pp]
            if beta is not None and len(buf) == pp:
                lag = np.array(buf, dtype=np.float64)[::-1]; pred = beta[0] + beta[1:] @ lag
                r_ = x - pred; self.arNss[pp] += r_ * r_; self.arNn[pp] += 1
            buf.append(x)
        for a in self.alphas:
            m0 = self.ew[a]; self.ew[a] = (1 - a) * m0 + a * x
            self.ewv[a] = (1 - a) * (self.ewv[a] + a * (x - m0) ** 2)
        for w, (buf, acc) in self.roll.items():
            if len(buf) == buf.maxlen:
                old = buf[0]; acc[0] -= old; acc[1] -= old * old
            buf.append(x); acc[0] += x; acc[1] += x * x
        self.specbuf.append(x)
        if len(self.specbuf) == self.SPEC_W:
            p = _norm_psd(np.array(self.specbuf, dtype=np.float64), self._spec_win)
            if p is not None: self._last_spec_ent, self._last_spec_cen = _spec_ec(p, self._spec_freqs, self._spec_logk)
        self.prev = x

        out = self._out; se = sd_h / math.sqrt(n)
        emp = np.cumsum(self.binc) / n
        ks = float(np.max(np.abs(emp - self._bin_targets)))
        if n > 2 and var > 0:
            mx = self.sx / n; cov = self.sxy / (n - 1) - mx * mx * n / (n - 1); ac1 = cov / var
        else: ac1 = 0.0
        vg = n >= self.VAR_MIN; sqrt_n = math.sqrt(n)
        out[0] = math.log1p(n); out[1] = (self.mean - mu_h) / max(se, 1e-8); out[2] = zx
        out[3] = math.log((var + 1e-8) / var_h) if vg else 0.0; out[4] = std / sd_h
        out[5] = (self.M3 / n) / (var ** 1.5 + 1e-8) if vg else 0.0
        out[6] = ((self.M4 / n) / (var ** 2 + 1e-8) - 3.0) if vg else 0.0
        out[7] = (self.cmax - self.cmin) / sqrt_n; out[8] = self.cusum_max / sqrt_n
        out[9] = self.tail2 / n; out[10] = self.tail3 / n
        out[11] = self.jumps / n; out[12] = self.signruns / n
        out[13] = ks; out[14] = self.rank_sum / n - 0.5; out[15] = ac1
        out[16] = ac1 - self.ref_ac1
        out[17] = (math.log((self.ar_resid_ss / self.ar_n + 1e-8) / (self.ar_var_h + 1e-8)) if self.ar_n >= self.VAR_MIN else 0.0)
        out[18] = self.ref_log_n; out[19] = self.ref_skew; out[20] = self.ref_kurt; out[21] = self.ref_ac1
        off = self._ew_off
        for ai, a in enumerate(self.alphas):
            ew_se = sd_h * math.sqrt(a / (2 - a))
            out[off + 2 * ai] = (self.ew[a] - mu_h) / max(ew_se, 1e-8)
            out[off + 2 * ai + 1] = math.log((self.ewv[a] + 1e-8) / var_h) if vg else 0.0
        off = self._roll_off
        for wi, w in enumerate(self.rw):
            buf, acc = self.roll[w]; L = len(buf); m = acc[0] / L; v = acc[1] / L - m * m
            if v < 0.0: v = 0.0
            out[off + 2 * wi] = (m - mu_h) / sd_h; out[off + 2 * wi + 1] = math.log((v + 1e-8) / var_h)
        off = self._spec_off
        out[off] = self._last_spec_ent; out[off + 1] = self._last_spec_ent - self.ref_spec_ent
        out[off + 2] = self._last_spec_cen - self.ref_spec_cen
        off = self._dep_off; j = 0; mean_all = self.mean
        for L in self.AC_LAGS_NEW:
            if self.acC[L] > 0 and var > 0 and vg:
                acL = (self.acP[L] / self.acC[L] - mean_all * mean_all) / var
            else: acL = 0.0
            out[off + j] = acL; out[off + j + 1] = acL - self.ref_ac[L]; j += 2
        for pp in self.AR_ORDERS_NEW:
            out[off + j] = (math.log((self.arNss[pp] / self.arNn[pp] + 1e-8) / (self.arN[pp][1] + 1e-8)) if self.arNn[pp] >= self.VAR_MIN else 0.0); j += 1
        eo = self._extra_off
        out[eo] = self.cusum_var_max / sqrt_n if vg else 0.0
        out[eo + 1] = float(np.max(emp - self._bin_targets) + np.max(self._bin_targets - emp))
        out[eo + 2] = float(np.mean((emp - self._bin_targets) ** 2))
        out[eo + 3] = out[5] - self.ref_skew if vg else 0.0
        out[eo + 4] = out[6] - self.ref_kurt if vg else 0.0
        go = self._glr_off                                                # GLR emit
        glr = self._var_glr(n) if (vg and n >= self.GLR_MIN) else 0.0
        if glr > self.var_glr_peak_v: self.var_glr_peak_v = glr
        out[go] = glr / n; out[go + 1] = self.var_glr_peak_v / n
        return out

def _log_steps(L, m):
    if L <= m: return np.arange(L)
    return np.unique(np.round(np.expm1(np.linspace(0, np.log1p(L - 1), m))).astype(int))


# ---- train / infer -------------------------------------------------------
def train(datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
          model_directory_path: str):
    params = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
                  bootstrap_type="Bernoulli", subsample=0.8, loss_function="Logloss",
                  random_seed=0, verbose=0, thread_count=-1)
    samples_per = 40

    st = StreamStateV3()
    cols = st.feature_names
    rows, labels, sids = [], [], []
    for dataset_id, x_hist, x_online, tau in datasets:
        xo = np.asarray(x_online, dtype=np.float64)
        st.reset(x_hist); L = len(xo)
        want = set(_log_steps(L, samples_per).tolist())
        for k in range(L):
            out = st.update(xo[k])
            if k not in want:
                continue
            rows.append(out.copy())
            labels.append(1 if (tau is not None and k >= tau) else 0)
            sids.append(dataset_id)

    Xtr = np.asarray(rows, np.float32); ytr = np.asarray(labels, np.int8)
    sids = np.asarray(sids)
    cmap = dict(zip(*np.unique(sids, return_counts=True)))
    w = np.array([1.0 / cmap[s] for s in sids]); w /= w.mean()
    model = CatBoostClassifier(**params).fit(Xtr, ytr, sample_weight=w)
    model.save_model(os.path.join(model_directory_path, "model.cbm"))
    joblib.dump({"cols": cols}, os.path.join(model_directory_path, "cols.joblib"))
    print("trained: %d rows, %d feats, %d series (CatBoost)" % (len(Xtr), len(cols), len(cmap)))


def infer(datasets: Iterable[Tuple[List[float], Iterable[float]]],
          model_directory_path: str):

    model = CatBoostClassifier()
    model.load_model(os.path.join(model_directory_path, "model.cbm"))
    cols = joblib.load(os.path.join(model_directory_path, "cols.joblib"))["cols"]

    st = StreamStateV3()
    if st.feature_names != cols:
        raise RuntimeError("feature schema drift: train vs infer StreamStateV3 differ")

    yield  # signal readiness

    for x_historical, x_online in datasets:
        st.reset(x_historical)
        for point in x_online:
            out = st.update(point)
            # fresh copy each step: CatBoost.predict marks its input array read-only
            # (zero-copy), which would clobber the reused StreamState._out buffer.
            row = np.array(out, dtype=np.float64).reshape(1, -1)
            yield float(model.predict_proba(row)[0, 1])


In [ ]:
# Keep-block: exported verbatim to the cloud script.
# Parallel workers fork after LightGBM's OpenMP is initialized in the parent —
# single worker is fine within budget.
# @crunch/keep:on
INFER_PARALLELISM = 1

# @crunch/keep:off

In [ ]:
# Local test + submit — MUST stay outside any @crunch/keep block.

# from sklearn.metrics import roc_auc_score
# crunch_tools.test()
# prediction = pd.read_parquet('prediction/prediction.parquet')
# y_test = pd.read_parquet('data/y_test.reduced.parquet')
# m = prediction.merge(y_test, how='left', left_index=True, right_index=True)
# m['t'] = m.groupby('id').cumcount()
# num = den = 0.0
# for t, g in m.groupby('t'):
#     yv = g['target'].values; sv = g['prediction'].values
#     npos = int(yv.sum()); nneg = len(yv) - npos
#     if npos == 0 or nneg == 0: continue
#     num += npos * nneg * roc_auc_score(yv, sv); den += npos * nneg
# print('Local TS-AUC: %.4f' % (num / den if den else 0.5))

# crunch_tools.submit(message='streaming GBM + spectral + expanded dependence, lr.02/1500 single')